# testConditionalGANPytorch1  
Andrew E. Davidson aedavids@ucsc.edu 8/29/24  

Copyright (c) 2020-2023, Regents of the University of California All rights reserved.   https://polyformproject.org/licenses/noncommercial/1.0.0

AIM: create a simple GAN that is easy to test our basic framework

generate y = x^2

ref: 
- chapter 6. in Generative Advisarial Networks with Python  
    this does not work. Keras/tensor flow version issues?  
    re-write example using pytorch

- [pytorch doc](https://pytorch.org/docs/stable/index.html)

In [1]:
import ipynbname
import matplotlib.pyplot as plt

from numpy import hstack
from numpy import zeros
from numpy import ones
from numpy.random import rand
from numpy.random import randn
import os

# by default keras use tensorflow as backend
import torch
print(f'torch.__version__: {torch.__version__}')

from torch import nn
torch.manual_seed(0) # Set for testing purposes, please do not change!

# tqdm provides progress bars for loops and iterables.
from tqdm.auto import tqdm

# class that helps you efficiently load and iterate over your dataset
# during training or inference
# we do not need this for our toy example
from torch.utils.data import DataLoader


notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

outDir = f'{notebookDir}/{notebookName}.out'
imgOut = f'{outDir}/img'
print(f'imgOut:\n{imgOut}')

torch.__version__: 2.5.1.post102
imgOut:
/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/gan/testConditionalGAN_pytorch_1.out/img


/private/home/aedavids/miniconda3/envs/extraCellularRNA/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Create models

In [2]:
class ParabolaGenerator( nn.Module ):
    def __init__( self, latentDim : int, nOutputs : int = 2 ) :
        '''
        arguments:
            latentDim: the length of the noise input vector

            nOutputs : the length of the output layers vector. Default = 2
        '''
        super().__init__()


        # keras
        # model.add(Dense(15, activation='relu', kernel_initializer='he_uniform', input_dim=latent_dim))
        # model.add(Dense(n_outputs, activation='linear'))
        
        self.model = nn.Sequential(
            nn.Linear(latentDim, 15),
            nn.ReLU(),
            nn.Linear(15, nOutputs),
        )
        # Initialize weights
        nn.init.kaiming_uniform_(self.model[0].weight, nonlinearity='relu')
        nn.init.zeros_(self.model[0].bias)
        nn.init.kaiming_uniform_(self.model[2].weight, nonlinearity='linear')
        nn.init.zeros_(self.model[2].bias)

    def forward( self, noise : torch.Tensor ):
        '''
            noise should be the value returned by generateLatentPoints()
        '''
        ret = self.model( noise )
        return ret

In [3]:
class ParabolaDiscriminator( nn.Module ):
    def __init__( self, inputSize : int = 2 ):
        '''
            inputSize, the length of the generated vectors
        '''
        super().__init__()

        # keras
        # model = Sequential()
    	# model.add(Dense(25, activation='relu', kernel_initializer='he_uniform', input_dim=n_inputs))
    	# model.add(Dense(1, activation='sigmoid'))
        
        self.model = nn.Sequential(
            nn.Linear(inputSize, 25),
            nn.ReLU(),
            nn.Linear(25, 1),
            nn.Sigmoid()
        )
        
        # Initialize weights
        nn.init.kaiming_uniform_(self.model[0].weight, nonlinearity='relu')
        nn.init.zeros_(self.model[0].bias)
        nn.init.kaiming_uniform_(self.model[2].weight, nonlinearity='sigmoid')
        nn.init.zeros_(self.model[2].bias)

    def forward( self, X : torch.Tensor ):
        ret = self.model( X )
        return ret

def testParabolaDiscriminator() :
    inputSize = 2
    discriminator = ParabolaDiscriminator( inputSize )
    numTest = 2
    testInput = torch.randn( numTest, inputSize )
    predictions = discriminator( testInput )
    print(f'\npredictions :\n{predictions}')

testParabolaDiscriminator()


predictions :
tensor([[0.5380],
        [0.5828]], grad_fn=<SigmoidBackward0>)


## Data Utilities
function to generate real and fake tensors

In [4]:
def generateRealSamples( n : int, device : str = 'cpu' ) -> tuple[torch.Tensor, torch.Tensor] :
    '''
    generate n real parabola samples with class labels

    arguments :
        n : number of examples to generated

        device. either 'cpu' or 'cuda', default 'cpu'

    Returns 2 Tensor
        X, y i.e. (realSamples, realLabels)

        y = 1, ie real
    '''
    # generate inputs in range [-0.5, 0.5]
    X1 = rand(n) - 0.5
    
    # generate outputs X^2
    X2 = X1 * X1
    
    # stack arrays
    X1 = X1.reshape(n, 1)
    X2 = X2.reshape(n, 1)
    X = hstack((X1, X2))
    
    # generate class labels
    y = ones((n, 1))

    # numpy dtype defaults to float64
    # pytorch default is float32
    realSamples = torch.tensor( X, dtype=torch.float32 ).to( device )
    realLabels = torch.tensor( y, dtype=torch.float32 ).to( device )
    
    return (realSamples, realLabels)

def testGenerateRealSamples() :
    X, y = generateRealSamples( n = 5 ) 
    print(f'X.device : {str(X.device)}')
    print( f'X.shape: {X.shape} rank : {len(X.shape)} num elements : {X.numel()}' )
    print( X )

    print(f'\ny.device : {str(y.device)}')
    print( f'y.shape: {y.shape} rank : {len(y.shape)} num elements : {y.numel()}' )
    print ( y) 

testGenerateRealSamples()

X.device : cpu
X.shape: torch.Size([5, 2]) rank : 2 num elements : 10
tensor([[ 0.4366,  0.1906],
        [-0.0724,  0.0052],
        [-0.4123,  0.1700],
        [ 0.3218,  0.1035],
        [-0.3192,  0.1019]])

y.device : cpu
y.shape: torch.Size([5, 1]) rank : 2 num elements : 5
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.]])


In [5]:
def generateLatentPoints(latentDimensions : int = 5, 
                         n : int = 100, device : str = 'cpu' ) -> torch.Tensor :
    '''
    generate points in latent space as input for the generator

    latentDimensions:
        the number of dimensions for the generator's input vector

    n the number of vectors to generate

    device. either 'cpu' or 'cuda', default 'cpu'


    returns a tensor
    '''
    # generate points in the latent space
    xInput = randn(latentDimensions * n)
    
    # reshape into a batch of inputs for the network
    xInput = xInput.reshape(n, latentDimensions)
    
    ret = torch.Tensor( xInput ).to( device )
    
    return ret

def testGenerateLatentPoints():
    tglp = generateLatentPoints(latentDimensions=5, n=3 )
    print(f'tglp.device : {str(tglp.device)}')
    print( tglp )
    print( tglp.shape )

testGenerateLatentPoints()

tglp.device : cpu
tensor([[ 0.4721,  0.0478, -0.3658,  1.1731,  0.5726],
        [ 1.1628, -1.8342, -0.5018, -0.6564, -1.7783],
        [-2.4104,  0.2185,  1.1247,  0.8836,  1.0051]])
torch.Size([3, 5])


In [6]:
def generateFakeSamples(generator : ParabolaGenerator, 
                        latentDimensions : int,
                        n : int,
                        device : str = 'cpu') -> tuple[torch.Tensor, torch.Tensor]:
    '''
    use the generator to generate n fake examples, with class labels

    gradient calculation is disabled. (should run faster)

    device. either 'cpu' or 'cuda', default 'cpu'


    returns (fakeSamples, fakeLabels)
        labels will be zeros.
    '''
    # generate points in latent space
    noiseVector = generateLatentPoints(latentDimensions, n)

    # Set the model to evaluation mode
    # Layers like Dropout and Batch Normalization behave differently during 
    # training and evaluation.
    #
    # we do not need to worry about this. It a good future proofing code example
    generator.eval()

    # Disable gradient calculation for inference
    with torch.no_grad():
        fakeSamples = generator(noiseVector)
    
    # create class labels  
    y = zeros((n, 1))
    fakeLabels = torch.Tensor( y ).to( device )
    
    return fakeSamples, fakeLabels

def testGenerateFakeSamples():
    latentDimensions = 5
    numOutputs = 2
    

    gen = ParabolaGenerator( latentDimensions, numOutputs )

    numSamples = 3
    fakeSamples, fakeLabels = generateFakeSamples( gen, latentDimensions, numSamples)

    print( f'\nfakeSamples.shape: {fakeSamples.shape} rank : {len(fakeSamples.shape)} num elements : {fakeSamples.numel()}' )
    print(f'fakeSamples.device : {str(fakeSamples.device)}')

    print( fakeSamples )

    print( f'\nfakeLabels.shape: {fakeLabels.shape} rank : {len(fakeLabels.shape)} num elements : {fakeLabels.numel()}' )
    print(f'fakeLabels.device : {str(fakeLabels.device)}')
    
    print ( fakeLabels ) 


testGenerateFakeSamples()


fakeSamples.shape: torch.Size([3, 2]) rank : 2 num elements : 6
fakeSamples.device : cpu
tensor([[-0.5064,  0.5732],
        [ 0.6269,  1.6854],
        [-0.0264,  1.5470]])

fakeLabels.shape: torch.Size([3, 1]) rank : 2 num elements : 3
fakeLabels.device : cpu
tensor([[0.],
        [0.],
        [0.]])


## Train Model

In [7]:
def getDiscriminatorLoss(generator, discriminator, criterion, realSamples, 
                         n, zDim, device) -> torch.Tensor:
    '''
    Return the loss of the discriminator on real and fake samples.
    
    Parameters:
        generator: the generator model, which returns an image given z-dimensional noise
        
        discriminator: the discriminator model, which returns a single-dimensional prediction of real/fake
        
        criterion: the loss function, which should be used to compare 
               the discriminator's predictions to the ground truth reality of the images 
               (e.g. fake = 0, real = 1)
               
        real: a batch of real samples
        
        n: the number of fakes the generator should produce, 
                which is also the length of the real images
                
        zDim: the dimension of the noise vector, a scalar
        
        device: the device type
        
    Returns:
        discriminatorLoss: a torch scalar loss value for the current batch
    '''
    # generate some fake samples
    fakeSamples, fakeLabels  = generateFakeSamples( generator, zDim, n, device )

    # get discriminator's prediction of the fake samples
    # forward
    predictionsOnFakes = discriminator( fakeSamples )
    
    # calculate the loss on fakes
    expected = torch.zeros(n, 1, device=device)  # All weights are equal to 0, ie fake
    discriminatorLossOnFake = criterion(predictionsOnFakes, expected) 
   
    # calculate loss on real samples
    predictionsOnReal = discriminator( realSamples )
    expected = torch.ones(n, 1, device=device)  # All weights are equal to 1, ie real
    discriminatorLossOnReal = criterion(predictionsOnReal, expected) 
   
    # calculate the average loss
    discriminatorLoss = (discriminatorLossOnReal + discriminatorLossOnFake) / 2
    
    return discriminatorLoss

In [8]:
def testGetDiscriminatorLossSample() :
    zDim = 5 # length of noise vector
    exampleLength = 2 # X,Y
    device = 'cpu'
    batchSize = 4

    generator= ParabolaGenerator(latentDim=zDim , nOutputs=exampleLength).to(device)    
    discriminator = ParabolaDiscriminator(inputSize=exampleLength).to(device) 
    criterion = nn.BCEWithLogitsLoss()

    realSamples, realLabels = generateRealSamples( batchSize, device )
    
    discriminatorLoss = getDiscriminatorLoss(generator, discriminator, criterion, 
                                             realSamples, batchSize,
                                              zDim, device)

    print( f'discriminator loss : {discriminatorLoss}') 

testGetDiscriminatorLossSample()

discriminator loss : 0.7077215909957886


In [9]:
def getGeneratorLoss(generator, discriminator, criterion, 
                     n, zDim, device) -> torch.Tensor:
    '''
    The generator never sees real samples. Loss is
    calculated only on fake samples

        Parameters:
        generator: the generator model, which returns an image given z-dimensional noise
        
        discriminator: the discriminator model, which returns a single-dimensional prediction of real/fake
        
        criterion: the loss function, which should be used to compare 
               the discriminator's predictions to the ground truth reality of the images 
               (e.g. fake = 0, real = 1)
               
        n: the number of fakes the generator should produce, 
                
        zDim: the dimension of the noise vector, a scalar
        
        device: the device type
        
    Returns:
        generatorLoss: a torch scalar loss value for the current batch
    '''
    noiseVector = generateLatentPoints(zDim, n, device)

    # forward() is the old way of doing things it breaks the grader
    #fakeImgs = gen.forward(noiseVector)
    fakeSamples = generator(noiseVector)
    
    # forward() is the old way of doing things     
    # predictionsOnFake = disc.forward( fakeImgs ).detach()
    predictionsOnFake = discriminator( fakeSamples )#.detach()
    
    # we are trying to trick the discriminator into thinking
    # the fakes are real
    expected = torch.ones(n, 1, device=device)  
    generatorLoss = criterion(predictionsOnFake, expected) 
    #     print(f'\nAEDWIP type(generatorLoss): {type(generatorLoss)}')
    #     print(f'AEDWIP generatorLoss.shape: {generatorLoss.shape}')
    #     print(f'AEDWIP generatorLoss: {generatorLoss}')   

    return generatorLoss

In [10]:
def testGetGeneratorLoss():
    zDim = 5 # length of noise vector
    n = 4 # number of fakes to generate  
    exampleLength = 2 # X,Y
    device = 'cpu'    
    
    generator= ParabolaGenerator(latentDim=zDim , nOutputs=exampleLength).to(device)    
    discriminator = ParabolaDiscriminator(inputSize=exampleLength).to(device)     
    criterion = nn.BCEWithLogitsLoss()

    generatorLoss = getGeneratorLoss(generator, discriminator, criterion, n, zDim, device) 
    print(f'generatorLoss.shape : {generatorLoss.shape}')
    print(f'generatorLoss\n{generatorLoss}')
    
testGetGeneratorLoss()    

generatorLoss.shape : torch.Size([])
generatorLoss
0.4824775457382202


In [11]:
# Set your parameters
criterion = nn.BCEWithLogitsLoss()
nEpochs = 2 # 200
numBatchs = 3 # number of batchs per epoch
zDim = 5 # length of noise vector
displayStep = 1 # 500
batchSize = 128
lr = 0.00001
exampleLength = 2 # X,Y
#device = 'cuda'
device = 'cpu'

generator= ParabolaGenerator(latentDim=zDim , nOutputs=exampleLength).to(device)
generatorOptimizer = torch.optim.Adam(generator.parameters(), lr=lr)

discriminator = ParabolaDiscriminator(inputSize=exampleLength).to(device) 
discriminatorOptimizer = torch.optim.Adam(discriminator.parameters(), lr=lr)

In [16]:
%%time
currentStep = 0
meanGeneratorLoss = 0
meanDiscriminatorLoss = 0
testGenerator = True # Whether the generator should be tested
genLoss = False
error = False


for epoch in range(nEpochs):
    print(f'epoch: {epoch}')    
    #  provides progress bars for loops and iterables.
    # will choose implementaiton based on env. ie shell, juypter, ...
    # https://tqdm.github.io/
    for i in tqdm( range(numBatchs) ) : 
        # if we where working with real data it is possible 
        # some batches are short. are < batchSize
        currentBatchSize = batchSize
        realSamples, realLabels = generateRealSamples( currentBatchSize )
        
        #
        # Update discriminator 
        #
        
        # Zero out the gradients before backpropagation
        discriminatorOptimizer.zero_grad()
        
        # Calculate discriminator loss
        discLoss = getDiscriminatorLoss(generator, discriminator, criterion, realSamples, 
                                  currentBatchSize, zDim, device)
        
        # Update gradients
        discLoss.backward(retain_graph=True)
        
        # Update optimizer
        discriminatorOptimizer.step()

        #
        # update generator
        #

        # For testing purposes, to keep track of the generator weights
        if testGenerator:
            # our last layer is a linear label not a list of layers
            #oldGeneratorWeights = generator.model[0][0].weight.detach().clone()
            oldGeneratorWeights = generator.model[0].weight.detach().clone()

        generatorOptimizer.zero_grad()
        generatorLoss = getGeneratorLoss(generator, discriminator, criterion, 
                                         currentBatchSize, zDim, device)
        generatorLoss.backward(retain_graph=True)
        generatorOptimizer.step()

print(f'finished')

epoch: 0


100%|███████████████████████████████████████████████████████| 3/3 [00:00<00:00, 152.26it/s]


epoch: 1


100%|███████████████████████████████████████████████████████| 3/3 [00:00<00:00, 154.15it/s]

finished
CPU times: user 3.03 s, sys: 183 ms, total: 3.21 s
Wall time: 46.5 ms


In [ ]:
# this is junk

# # create a discrimanator
# discriminatorModel = ParabolaDiscriminator( inputSize=2 )
# # define the cost function
# discriminatorCriterion = nn.BCELoss()

# # use stochastic gradient decent
# # keras
# # 	discrModel.compile(loss='binary_crossentropy', optimizer='adam', 
# # metrics=['accuracy'])

# learningRate = 0.01
# discriminatorOptimizer = torch.optim.adam( discriminatorModel.parameters, 
#                                           lr=learningRate )

# # define the number of training loops
# numEpochs = aedwip
# for t in range( numEpochs ) :

#     # forward propagation
#     # get a prediction
#     yHat = discriminatorModel( X )
#     discriminatorLoss = discriminatorCriterion( yHat, y)

#     # backward propagation
#     discriminatorOptimizer.zero_grad()
#     discriminatorLoss.backward()

#     # update the parameters
#     discriminatorOptimizer.step()
    